# 01. Границы Малой Охты: четыре версии

Приём проекта «Коломна»: у района нет одной границы, поэтому аналитика ведётся по **четырём вложенным версиям**. Каждая цифра диагностики в документах размечается по версии границ.

| версия | что это | принцип |
|---|---|---|
| v1 | историческое ядро Малой Охты | прибрежные кварталы у устья Охты: авторский полигон по берегам Невы и Охты, воспроизводим из данных |
| v2 | МО Малая Охта | официальная граница (OSM, relation 1185381) |
| v3 | охтинский полуостров с мысом | клин между Невой и рекой Охта: историческая территория Ниеншанца |
| v4 | вся административная Охта | МО Малая Охта + МО Большая Охта (relations 1185381 + 1185383) |

Источники: Nominatim lookup (admin polygons), Overpass (вода). Все запросы кэшируются в `data/raw/cache/` — ноутбук работает офлайн на готовых выборках.

In [1]:
# Настройка: пути, константы. Проект рассчитан на запуск из корня репозитория.
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, LineString, Polygon
from shapely.ops import unary_union

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
RAW, PROC, CACHE = PROJECT / "data/raw", PROJECT / "data/processed", PROJECT / "data/raw/cache"
PROC.mkdir(parents=True, exist_ok=True)

CRS_STORE = 4326      # хранение
CRS_METRIC = 32636    # метрика (UTM 36N), см. AGENTS.md
print("проект:", PROJECT)

проект: /home/mfthesun/Projects/malaya-okhta-strategy


In [2]:
# v2 и v4: официальные границы округов из кэша Nominatim lookup
_cache = json.load(open(CACHE / "nominatim_mo_boundaries.json", encoding="utf-8"))
_rows = []
for item in _cache:
    rows_ = {"osm_id": item["osm_id"], "name": item["display_name"].split(",")[0].strip(),
             "geometry": shape(item["geojson"])}
    _rows.append(rows_)
mo = gpd.GeoDataFrame(_rows, crs="EPSG:4326").set_index("name")
print(mo)

v2 = mo.loc["округ Малая Охта", "geometry"]
v4 = unary_union(mo.geometry)
mo_wgs = mo.copy()
mo["area_km2"] = mo.to_crs(CRS_METRIC).geometry.area / 1e6
print("\nплощади, км2:")
print(mo[["area_km2"]].round(2))

                     osm_id                                           geometry
name                                                                          
округ Малая Охта    1185381  POLYGON ((30.39428 59.92902, 30.39591 59.92574...
округ Большая Охта  1185383  POLYGON ((30.39958 59.95544, 30.40216 59.95384...

площади, км2:
                    area_km2
name                        
округ Малая Охта        5.26
округ Большая Охта      5.80


In [7]:
# Вода из кэша Overpass: осевые линии Невы и Охты (riverbank-полигоны в bbox — ways без тегов внутри relations,
# поэтому работаем с осями; полигональные берега для выреза акваторий возьмём на этапе 02 из слоя water)
from shapely.ops import nearest_points

wp = json.load(open(CACHE / "overpass_water_bbox.json", encoding="utf-8"))
lines = []
for el in wp["elements"]:
    if "geometry" not in el:
        continue
    tags = el.get("tags", {})
    if el["type"] == "way" and tags.get("waterway") == "river":
        lines.append({"name": tags.get("name", ""), "id": el["id"],
                      "geometry": LineString([(g["lon"], g["lat"]) for g in el["geometry"]])})
water_line = gpd.GeoDataFrame(lines, geometry="geometry", crs="EPSG:4326")
print("осевые линии:", water_line.name.value_counts().to_dict())

# Ключевой ориентир: устье Охты — ближайшая точка осей Невы и Охты
neva = water_line[water_line.name == "Нева"].geometry.union_all()
okhta = water_line[water_line.name == "Охта"].geometry.union_all()
mouth = nearest_points(neva, okhta)[0]
print("устье Охты ~", round(mouth.x, 4), round(mouth.y, 4))
print("bbox v2:", [round(v, 4) for v in v2.bounds])

осевые линии: {'Нева': 5, 'Охта': 3, 'Лапка': 3, '': 3, 'Монастырка': 1, 'Чернавка': 1, 'Лубья': 1, 'Оккервиль': 1}
устье Охты ~ 30.403 59.9486
bbox v2: [30.3943, 59.9154, 30.4486, 59.9486]


In [8]:
# Версии границ:
#   v1 — историческое ядро: часть МО Малая Охта в 600 м от устья Охты (Охтинский мыс);
#   v2 — МО Малая Охта (официальная граница);
#   v3 — вся административная Охта: МО Малая Охта + МО Большая Охта;
#   v4 — охтинский полуостров: суша в пределах 1 200 м от осей Невы и Охты
#        (историко-географический ареал Ниеншанца; акватории в кэше только осями —
#         после появления полигонального слоя water на этапе 02 v4 пересчитывается).
from shapely.ops import nearest_points

# буфер в метрах: в UTM 36N и обратно в EPSG:4326
def metric_buffer(geom, dist_m):
    return gpd.GeoSeries([geom], crs=4326).to_crs(CRS_METRIC).buffer(dist_m).to_crs(4326).iloc[0]

v1 = v2.intersection(metric_buffer(mouth, 600))
v3 = mo.geometry.union_all()

rivers = neva.union(okhta)
v4 = v3.intersection(metric_buffer(rivers, 1200))

versions = gpd.GeoDataFrame(
    [{"version": "v1", "title": "Историческое ядро (600 м от мыса)", "geometry": v1},
     {"version": "v2", "title": "МО Малая Охта", "geometry": v2},
     {"version": "v3", "title": "Вся административная Охта (2 округа)", "geometry": v3},
     {"version": "v4", "title": "Охтинский полуостров (1,2 км от осей рек)", "geometry": v4}],
    geometry="geometry", crs="EPSG:4326")
versions["area_km2"] = versions.to_crs(CRS_METRIC).geometry.area / 1e6
print(versions[["version", "title", "area_km2"]].round(2).to_string(index=False))

print("\nвложенность:")
for a, b in [("v1", "v2"), ("v2", "v3")]:
    ga = versions.loc[versions.version == a, "geometry"].iloc[0]
    gb = versions.loc[versions.version == b, "geometry"].iloc[0]
    print(f"  {a} в {b}: {100 * ga.intersection(gb).area / ga.area:.1f} %")
print(f"  v4 в v3: {100 * v4.intersection(v3).area / v4.area:.1f} %")
print(f"  v1 в v4: {100 * v1.intersection(v4).area / v1.area:.1f} %")

versions.to_file(PROC / "boundary_versions.geojson", driver="GeoJSON")
print("\nсохранено:", PROC / "boundary_versions.geojson")

version                                     title  area_km2
     v1         Историческое ядро (600 м от мыса)      0.15
     v2                             МО Малая Охта      5.26
     v3      Вся административная Охта (2 округа)     11.06
     v4 Охтинский полуостров (1,2 км от осей рек)     10.55

вложенность:
  v1 в v2: 100.0 %
  v2 в v3: 100.0 %
  v4 в v3: 100.0 %
  v1 в v4: 100.0 %

сохранено: /home/mfthesun/Projects/malaya-okhta-strategy/data/processed/boundary_versions.geojson


In [9]:
# Объекты культурного наследия по версиям: ЕГРОКН с координатами внутри границ
okn = pd.read_csv(RAW / "egrokn_spb_all.csv", encoding="utf-8-sig")
okn_g = gpd.GeoDataFrame(
    okn.dropna(subset=["lon", "lat"]), geometry=gpd.points_from_xy(okn.dropna(subset=["lon", "lat"])["lon"],
                                                                    okn.dropna(subset=["lon", "lat"])["lat"]),
    crs="EPSG:4326")
print(f"ЕГРОКН по СПб: {len(okn)} записей, из них с координатами: {len(okn_g)}")

for _, row in versions.iterrows():
    inside = okn_g[okn_g.within(row.geometry)]
    cat = inside["cat"].value_counts().to_dict()
    print(f"{row['version']} ({row['title']}): {len(inside)} ОКН, категории: {cat}")

print("\nОКН внутри v3, содержащие «Охта»/«Ниеншанц» в названии:")
v3_geom = versions.loc[versions.version == "v3", "geometry"].iloc[0]
names = okn_g[okn_g.within(v3_geom)]
names = names[names["name"].str.contains("Охта|Ниеншанц", case=False, na=False)]
for _, r in names[["name", "address"]].iterrows():
    print("  ", r["name"][:75], "|", str(r["address"])[:40])

ЕГРОКН по СПб: 6987 записей, из них с координатами: 6401
v1 (Историческое ядро (600 м от мыса)): 0 ОКН, категории: {}
v2 (МО Малая Охта): 21 ОКН, категории: {'Регионального значения': 16, 'Федерального значения': 5}
v3 (Вся административная Охта (2 округа)): 59 ОКН, категории: {'Регионального значения': 43, 'Федерального значения': 16}
v4 (Охтинский полуостров (1,2 км от осей рек)): 59 ОКН, категории: {'Регионального значения': 43, 'Федерального значения': 16}

ОКН внутри v3, содержащие «Охта»/«Ниеншанц» в названии:


In [10]:
# Население по версиям: ряд Петростата по МО (v2, v3); для v1 и v4 — оценка по слоям застройки на этапе 03
pop = pd.read_csv(RAW / "population_spb_mo_2010_2025.csv", encoding="utf-8-sig")
show = pop[pop["территория"].str.contains("Охта|Красногвардейский", na=False)][
    ["территория", "ВПН-2010", "01.01.2020", "ВПН-2020 (01.10.2021)", "01.01.2025"]]
print(show.to_string(index=False))

pop_mal = pop.loc[pop["территория"] == "МО Малая Охта"].iloc[0]
pop_bol = pop.loc[pop["территория"] == "МО Большая Охта"].iloc[0]
print(f"\nv2 (МО Малая Охта, 01.01.2025): {int(pop_mal['01.01.2025'])} чел.")
print(f"v3 (вся Охта, 01.01.2025): {int(pop_mal['01.01.2025'] + pop_bol['01.01.2025'])} чел.")
print(f"v1, v4: численность оценивается по слоям застройки (ноутбук 03)")

             территория  ВПН-2010  01.01.2020  ВПН-2020 (01.10.2021)  01.01.2025
Красногвардейский район    337091      356628                 369347      368354
        МО Большая Охта     54539       56658                  58616       58176
          МО Малая Охта     45946       47703                  43849       43319

v2 (МО Малая Охта, 01.01.2025): 43319 чел.
v3 (вся Охта, 01.01.2025): 101495 чел.
v1, v4: численность оценивается по слоям застройки (ноутбук 03)


## Выводы ноутбука 01 (прогон 02.11.2026)

1. **Четыре версии границ** воспроизведены и сохранены в `data/processed/boundary_versions.geojson`:
   v1 ядро 0,15 км² → v2 МО Малая Охта 5,26 км² → v3 вся Охта 11,06 км²; v4 полуостров 10,55 км². Вложенность v1⊂v2⊂v3 — 100 %.
2. **Население** (Петростат/ВПН): МО Малая Охта 45 946 (2010) → 43 319 (01.01.2025), −5,7 %; вся Охта (2 округа) — 101 495 чел.
3. **ОКН** (ЕГРОКН с координатами): v2 — 21 (5 фед., 16 рег.), v3/v4 — 59 (16 фед., 43 рег.).
4. **Оговорка**: у Ниеншанца (Охта 1) в выгрузке ЕГРОКН нет ни адреса, ни координат — привязывается вручную к Охтинскому мысу на этапе 02 (словарь MANUAL по образцу Коломны). Часть ОКН в v1 может не учитываться до этой привязки.
5. v4 построен по буферу 1 200 м от осей рек; после появления полигонального слоя воды (этап 02) пересчитывается — ожидаемое сужение на акватории Невы и Охты.

Роль версий в стратегии: v1 — ядро атаки (флагманские проекты), v2 — базовая аналитическая единица (данные по населению), v3 — транспортная связность и сервисы, v4 — историко-культурный ареал (Ниеншанц, береговая линия).